# 🎭 FreeFakeStudio (Kaggle Edition)
**Multi-Model Studio for Kaggle GPU** — Generate · Img2Img · Inpaint

### ⚠️ Before Running (Kaggle Setup):
1. Look at the right sidebar under **Session options**:
   - **Accelerator**: Select **GPU T4 x2** or **GPU P100** *(Do NOT use TPU — ComfyUI/CUDA requires an NVIDIA GPU)*.
   - **Internet**: Toggle to **Internet on** *(Required to download models & create the Gradio link)*.
2. Run the cells in order: **Cell 1** ➔ **Cell 2** ➔ **Cell 3**.

In [ ]:
# @title 🎯 Select Models to Download
# @markdown Run this cell to display the interactive model selector checkboxes!

import ipywidgets as widgets
from IPython.display import display, HTML

# ── Interactive Checkboxes ──────────────────────────────────
cb_z_image = widgets.Checkbox(
    value=True,
    description='⚡ Z-Image Turbo — Fast generation (8 steps, ~1 min) [Default & Recommended]',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', margin='3px 0')
)
cb_flux_4b = widgets.Checkbox(
    value=False,
    description='🌊 FLUX.2-klein 4B — Photorealistic generation & inpainting',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', margin='3px 0')
)
cb_flux_9b = widgets.Checkbox(
    value=False,
    description='🔮 FLUX.2-klein 9B — Ultra-detail high-res generation & editing',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', margin='3px 0')
)
cb_qwen = widgets.Checkbox(
    value=False,
    description='🎨 Qwen-Image-Edit — Conversational instruction image editor',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', margin='3px 0')
)
cb_ernie = widgets.Checkbox(
    value=False,
    description='🖌️ ERNIE-Image-Turbo — Fast text-to-image (Baidu, 8 steps)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', margin='3px 0')
)

# Quick Select Buttons
btn_fast = widgets.Button(description="⚡ Fast Only", button_style="success", layout=widgets.Layout(width="110px", height="30px"))
btn_all = widgets.Button(description="✨ Select All", layout=widgets.Layout(width="110px", height="30px"))

def on_fast_clicked(b):
    cb_z_image.value = True
    cb_flux_4b.value = False
    cb_flux_9b.value = False
    cb_qwen.value = False
    cb_ernie.value = False

def on_all_clicked(b):
    cb_z_image.value = True
    cb_flux_4b.value = True
    cb_flux_9b.value = True
    cb_qwen.value = True
    cb_ernie.value = True

btn_fast.on_click(on_fast_clicked)
btn_all.on_click(on_all_clicked)

display(HTML('''
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; margin-bottom: 6px;">
    <h3 style="color: #047857; margin: 0 0 4px 0; font-size: 18px; font-weight: 800;">🎯 Select Models to Download</h3>
    <p style="color: #4b5563; margin: 0 0 10px 0; font-size: 13px;">Check the models you want to use below (Z-Image Turbo is selected by default):</p>
</div>
'''))

btn_bar = widgets.HBox([btn_fast, btn_all], layout=widgets.Layout(margin="0 0 8px 0"))
chk_box = widgets.VBox([cb_z_image, cb_flux_4b, cb_flux_9b, cb_qwen, cb_ernie])

card = widgets.VBox([btn_bar, chk_box], layout=widgets.Layout(
    padding='14px 18px',
    border='2px solid #10b981',
    border_radius='12px',
    background_color='#f0fdf4'
))
display(card)



In [ ]:
# @title ⬇️ Step 1: Install & Download Selected Models
import os, sys, subprocess, glob
import torch

# Read model selection from interactive checkboxes (or fallback to defaults)
try:
    Z_Image_Turbo = cb_z_image.value
    FLUX_Klein_4B = cb_flux_4b.value
    FLUX_Klein_9B = cb_flux_9b.value
    Qwen_Image_Edit = cb_qwen.value
    ERNIE_Image_Turbo = cb_ernie.value
except NameError:
    Z_Image_Turbo = True
    FLUX_Klein_4B = False
    FLUX_Klein_9B = False
    Qwen_Image_Edit = False
    ERNIE_Image_Turbo = False

selected_models = [m for m, val in [
    ('⚡ Z-Image Turbo', Z_Image_Turbo),
    ('🌊 FLUX.2-klein 4B', FLUX_Klein_4B),
    ('🔮 FLUX.2-klein 9B', FLUX_Klein_9B),
    ('🎨 Qwen-Image-Edit', Qwen_Image_Edit),
    ('🖌️ ERNIE-Image Turbo', ERNIE_Image_Turbo),
] if val]

print(f"📦 Selected Model(s) to Install: {', '.join(selected_models)}")

if not torch.cuda.is_available():
    print("❌ ERROR: No GPU detected!")
    print("   Please go to Session options on the right sidebar, change Accelerator to 'GPU T4 x2', and restart.")
    raise SystemExit(1)

import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=5)
except Exception:
    print("❌ ERROR: Internet access is OFF!")
    print("   Please go to Session options on the right sidebar -> toggle 'Internet on'.")
    raise SystemExit(1)

if not any([Z_Image_Turbo, FLUX_Klein_4B, FLUX_Klein_9B, Qwen_Image_Edit, ERNIE_Image_Turbo]):
    print("❌ No models selected! Go back to Cell 1, check at least one model, and run it again.")
    raise SystemExit(1)

import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=5)
except Exception:
    print("❌ ERROR: Internet access is OFF!")
    print("   Please go to Session options on the right sidebar -> toggle 'Internet on'.")
    raise SystemExit(1)

print("=" * 60)
print(f"🚀 Setting up FreeFakeStudio on Kaggle GPU ({torch.cuda.get_device_name(0)})")
print("=" * 60)

# ── Ensure /content symlink for cross-platform compatibility ──
os.makedirs('/content', exist_ok=True)
subprocess.run(["ln", "-sfn", "/kaggle/working/ComfyUI", "/content/ComfyUI"], check=False)
subprocess.run(["ln", "-sfn", "/kaggle/working/FreeFakeStudio", "/content/FreeFakeStudio"], check=False)

# ── 1. ComfyUI Core ─────────────────────────────────────────
print("\n⏳ [1/4] Installing ComfyUI Core...")
if not os.path.exists('/kaggle/working/ComfyUI'):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", "/kaggle/working/ComfyUI"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/kaggle/working/ComfyUI/requirements.txt"], check=True)
print("  ✅ ComfyUI Core ready!")

# ── ComfyUI-GGUF (needed for FLUX 9B, Qwen Edit, ERNIE) ───────
if (Qwen_Image_Edit or FLUX_Klein_9B or ERNIE_Image_Turbo) and not os.path.exists('/kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    print("  ⏳ Installing ComfyUI-GGUF support...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/city96/ComfyUI-GGUF.git", "/kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gguf"], check=True)

# ── Diffusers (needed for Qwen VAE) ─────────────────────────
if Qwen_Image_Edit:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers>=0.31.0"], check=True)

# ── 2. Studio Utilities & Dependencies ──────────────────────
print("\n⏳ [2/4] Installing Studio Dependencies...")
subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "aria2"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rembg[gpu]", "onnxruntime-gpu", "aria2", "gradio"], check=True)
print("  ✅ Dependencies ready!")

# ── 3. Model Downloads ──────────────────────────────────────
print("\n⏳ [3/4] Downloading Selected Models...")
DIFF   = '/kaggle/working/ComfyUI/models/diffusion_models'
CLIP   = '/kaggle/working/ComfyUI/models/clip'
TXTENC = '/kaggle/working/ComfyUI/models/text_encoders'
VAE    = '/kaggle/working/ComfyUI/models/vae'
for d in [DIFF, CLIP, TXTENC, VAE]:
    os.makedirs(d, exist_ok=True)

def dl(url, dest_dir, name):
    path = os.path.join(dest_dir, name)
    if not (os.path.exists(path) and os.path.getsize(path) > 1024):
        if os.path.exists(path):
            os.remove(path)
        print(f"  ⬇️ Downloading {name}...")
        subprocess.run(["aria2c", "--console-log-level=warn", "-c", "-x", "16", "-s", "16", "-k", "1M", url, "-d", dest_dir, "-o", name], check=True)
        print(f"  ✅ {name} downloaded!")
    else:
        print(f"  ✅ {name} already cached!")

if Z_Image_Turbo:
    print("\n📦 Z-Image Turbo FP8:")
    dl('https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors', DIFF, 'z-image-turbo-fp8-e4m3fn.safetensors')
    dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors', CLIP, 'qwen_3_4b.safetensors')
    dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors', VAE, 'ae.safetensors')

if FLUX_Klein_9B or FLUX_Klein_4B:
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-9b/resolve/main/split_files/vae/flux2-vae.safetensors', VAE, 'flux2-vae.safetensors')

if FLUX_Klein_4B:
    print("\n📦 FLUX.2-klein 4B:")
    dl('https://huggingface.co/black-forest-labs/FLUX.2-klein-4B/resolve/main/flux-2-klein-4b.safetensors', DIFF, 'flux-2-klein-4b.safetensors')
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/text_encoders/qwen_3_4b_fp4_flux2.safetensors', TXTENC, 'qwen_3_4b_fp4_flux2.safetensors')

if FLUX_Klein_9B:
    print("\n📦 FLUX.2-klein 9B:")
    dl('https://huggingface.co/black-forest-labs/FLUX.2-klein-9b-kv-fp8/resolve/main/flux-2-klein-9b-kv-fp8.safetensors', DIFF, 'flux-2-klein-9b-kv-fp8.safetensors')
    dl('https://huggingface.co/unsloth/Qwen3-8B-GGUF/resolve/main/Qwen3-8B-Q2_K_L.gguf', TXTENC, 'Qwen3-8B-Q2_K_L.gguf')

if Qwen_Image_Edit:
    print("\n📦 Qwen-Image-Edit-2511:")
    dl('https://huggingface.co/unsloth/Qwen-Image-Edit-2511-GGUF/resolve/main/qwen-image-edit-2511-Q3_K_M.gguf', DIFF, 'qwen-image-edit-2511-Q3_K_M.gguf')
    dl('https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/Qwen2.5-VL-7B-Instruct-Q3_K_M.gguf', CLIP, 'Qwen2.5-VL-7B-Instruct-Q3_K_M.gguf')
    dl('https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/mmproj-F16.gguf', CLIP, 'Qwen2.5-VL-7B-Instruct-mmproj-F16.gguf')
    dl('https://huggingface.co/Qwen/Qwen-Image-Edit-2511/resolve/main/vae/diffusion_pytorch_model.safetensors', VAE, 'qwen_image_vae.safetensors')

if ERNIE_Image_Turbo:
    print("\n📦 ERNIE-Image-Turbo:")
    dl('https://huggingface.co/unsloth/ERNIE-Image-Turbo-GGUF/resolve/main/ernie-image-turbo-Q6_K.gguf', DIFF, 'ernie-image-turbo-Q6_K.gguf')
    dl('https://huggingface.co/Comfy-Org/ERNIE-Image/resolve/main/text_encoders/ministral-3-3b.safetensors', TXTENC, 'ministral-3-3b.safetensors')
    if not (FLUX_Klein_9B or FLUX_Klein_4B):
        dl('https://huggingface.co/Comfy-Org/ERNIE-Image/resolve/main/vae/flux2-vae.safetensors', VAE, 'flux2-vae.safetensors')

for f in glob.glob(f'{CLIP}/*'):
    link = os.path.join(TXTENC, os.path.basename(f))
    if not os.path.exists(link):
        try: os.symlink(f, link)
        except: pass
for f in glob.glob(f'{TXTENC}/*'):
    link = os.path.join(CLIP, os.path.basename(f))
    if not os.path.exists(link):
        try: os.symlink(f, link)
        except: pass

print("\n🎉 Step 1 Complete! All selected models and dependencies are ready.")



In [ ]:
# @title 🚀 Step 2: Launch FreeFakeStudio
import os
os.chdir('/kaggle/working')
REPO = 'https://github.com/MuntasirMalek/FreeFakeStudio.git'
if os.path.exists('/kaggle/working/FreeFakeStudio'):
    %cd /kaggle/working/FreeFakeStudio
    !git pull --ff-only || true
else:
    !git clone --depth 1 {REPO} /kaggle/working/FreeFakeStudio
    %cd /kaggle/working/FreeFakeStudio

!python app.py

